In [ ]:
# ============================================================
# CardI-HACK - Project
# ============================================================

In [ ]:
import pandas as pd
import numpy as np

print("Data loading...")

train = pd.read_csv("path to cardihack_final-train.csv")
test  = pd.read_csv("path to cardihack_final-test.csv")
dummy = pd.read_csv("path to cardihack_dummy_pred.csv") #  to compare the submission file

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print("Dummy columns:", dummy.columns.tolist())

ID_COL = "trustii_id"
AGE_COL = "Age_Baseline"
SEX_COL = "Genre"

TARGET_SEVERITY = "OUTCOME SEVERITY"
TARGET_MACE     = "OUTCOME MACE" 

SNP_COLS = [c for c in train.columns if c.startswith("SNP")]  


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from scipy.stats import chi2_contingency

def chi2_select(df, target):
    selected = [] 
    for snp in SNP_COLS:
        t = pd.crosstab(df[snp], df[target])
        if t.shape[0] > 1:
            _, p, _, _ = chi2_contingency(t)
            if p < 0.05:
                selected.append(snp)
    return selected

sev_snps  = chi2_select(train, TARGET_SEVERITY)
mace_snps = chi2_select(train, TARGET_MACE)

# ----- SEVERITY (proba)
sev_features = [AGE_COL, SEX_COL] + sev_snps

sev_model = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, random_state=42))
])

sev_model.fit(train[sev_features], train[TARGET_SEVERITY])
severity_pred = sev_model.predict_proba(test[sev_features])[:, 1]

# ----- MACE (classe)
mace_features = [AGE_COL, SEX_COL] + mace_snps

mace_model = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
    ("lr", LogisticRegression(
        multi_class="multinomial",
        max_iter=2000,
        random_state=42
    ))
])

mace_model.fit(train[mace_features], train[TARGET_MACE])
mace_pred = mace_model.predict(test[mace_features])

# ----- SUBMISSION (FORMAT DUMMY)
submission = pd.DataFrame({
    "trustii_id": test[ID_COL],
    "OUTCOME MACE": mace_pred.astype(int),
    "OUTCOME SEVERITY": severity_pred.astype(float)
})

submission = submission[dummy.columns]
submission.to_csv("submission_logistic.csv", index=False)
print("submission_logistic.csv saved. Final shape:", submission.shape)


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from scipy.stats import chi2_contingency

df = pd.read_csv(r"cardihack_final-train.csv")
print(df.shape)
df.head()

priority_snps = [f"SNP{i}" for i in range(1, 76)]
X_snps = df[priority_snps].copy()
#handle missing values
X_snps = X_snps.fillna(X_snps.median())

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_snps)

#silhouette analysis
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sil_scores = []

for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42)
    labels = km.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels))

for k, s in zip(range(2, 8), sil_scores):
    print(f"k={k}, silhouette={s:.3f}")

- We have silhouette scores ranging from -1 to 1.
- We note that scores higher than 0.5 indicates strong clustering and when close to 0, it is indicative of no meaningful cluster structure.
- Here, the k with highest silhouette score is 0,019 for k=2.

-> All scores are close to 0 and even the best one at k=2, indicating no natural clustering structure in the SNP data.

In [ ]:
contingency_mace = pd.crosstab(df["Cluster"], df["OUTCOME MACE"])
print(contingency_mace)

The outcome distributions are differents across clusters, it may be similar for cluster 0 & 2.
Differences are seen for cluster 1 and clusters 0 & 2.

In [ ]:
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(contingency_mace)
print(f"Chi2 p-value (clusters vs MACE): {p:.4f}")

- Chi2 pvalue is bigger than 0.05, outcome mace is statistically independant of cluster.
- Clusters do not stratify mace in any significant way.

In [ ]:
contingency_sev = pd.crosstab(df["Cluster"], df["OUTCOME SEVERITY"])
print(contingency_sev)

chi2, p, dof, expected = chi2_contingency(contingency_sev)
print(f"Chi2 p-value (clusters vs Severity): {p:.4f}")

- Chi2 pvalue is bigger than 0.05, outcome severity is statistically independant of cluster.
- Clusters do not stratify severity in any significant way.

Conclusion of k-means:
- Clusters don't explain MACE & severity.
- K-means expects spherical and well-seperated clusters but SNP data is noisy.

In [ ]:
# ============================================================
# K-MEANS → CLUSTER-RISK PREDICTION MODEL
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

features = [AGE_COL, SEX_COL] + SNP_COLS

# ----- K-MEANS PIPELINE
km_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
    ("km", KMeans(n_clusters=3, n_init=10, random_state=42))
])

# ----- FIT ON TRAIN
train["Cluster"] = km_pipe.fit_predict(train[features])

# ----- LEARN CLUSTER RISKS (SUPERVISED STEP)
cluster_sev_risk = train.groupby("Cluster")[TARGET_SEVERITY].mean()
cluster_mace_risk = train.groupby("Cluster")[TARGET_MACE].mean()

# ----- APPLY TO TEST
test["Cluster"] = km_pipe.predict(test[features])

severity_pred = test["Cluster"].map(cluster_sev_risk)
mace_pred = test["Cluster"].map(cluster_mace_risk)

# ----- SUBMISSION
submission = pd.DataFrame({
    "trustii_id": test[ID_COL],
    "OUTCOME MACE": (mace_pred > 0.5).astype(int),
    "OUTCOME SEVERITY": severity_pred
})

submission = submission[dummy.columns]
submission.to_csv("submission_kmeans.csv", index=False)

print("✅ submission_kmeans.csv saved")


In [ ]:
priority_snps = [f"SNP{i}" for i in range(1, 76)]
X_snps = df[priority_snps].copy()

#missing values
X_snps = X_snps.fillna(X_snps.median())

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_snps)

#Ward linkage
from sklearn.cluster import AgglomerativeClustering

hc = AgglomerativeClustering(
    n_clusters=3,      # start with 2–6, tune later
    linkage="ward"
)

clusters = hc.fit_predict(X_scaled)
df["Cluster"] = clusters

from sklearn.metrics import silhouette_score

for k in range(2, 7):
    hc = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = hc.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    print(f"k={k}, silhouette={score:.3f}")


- Silhouette scores ranges from -1 to 1.
- We note that scores higher than 0.5 indicates strong clustering and when close to 0, it is indicative of no meaningful cluster structure.
- Here, the k with highest silhouette score is 0,126 for k=2.

-> All scores are close to 0 and even the best one at k=2, indicating no natural clustering structure in the SNP data.

In [ ]:
contingency_mace = pd.crosstab(df["Cluster"], df["OUTCOME MACE"])
print(contingency_mace)

The outcome distributions are differents across clusters, it may be similar for cluster 0 & 2.
Differences are seen for cluster 1 and clusters 0 & 2.

In [ ]:
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(contingency_mace)
print(f"Chi-square p-value (Cluster vs MACE): {p:.4f}")

p < 0.05 → genetic clusters differ in MACE risk

p ≥ 0.05 → no genetic stratification

- Chi2 pvalue is bigger than 0.05, outcome mace is statistically independant of cluster, no genetic stratification.
- Clusters do not stratify patients by mace risk.

In [ ]:
contingency_sev = pd.crosstab(df["Cluster"], df["OUTCOME SEVERITY"])
print(contingency_sev)

chi2, p, dof, expected = chi2_contingency(contingency_sev)
print(f"Chi-square p-value (Cluster vs Severity): {p:.4f}")

- Chi2 pvalue is bigger than 0.05, outcome severity is statistically independant of cluster.
- Clusters do not stratify severity in any significant way.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

linked = linkage(X_scaled, method="ward")

plt.figure(figsize=(12, 6))
dendrogram(linked, truncate_mode="level", p=5)
plt.title("Hierarchical clustering dendrogram (Ward linkage)")
plt.xlabel("Samples")
plt.ylabel("Distance")
plt.show()

- We have no hierarchical separation of patients, no discrete subpopulations.
-> The dendogram shows us weak hierarchical structure.

Conclusion of hierarchical clustering:
- Hierarchical slutering of priority SNPs did not reveal defined gentic subgroups, because we have low sihouette scores.
- Clusters aren't significantly associated with either MACE or disease severity.
- Maybe the use of supervised associations or prediction models over unsupervised clustering approaches may be more appropriate.

In [ ]:
# ============================================================
# HIERARCHICAL CLUSTERING → CLUSTER-RISK PREDICTION MODEL
# ============================================================

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from scipy.spatial.distance import cdist

features = [AGE_COL, SEX_COL] + SNP_COLS

# ----- PREPROCESS
imp = SimpleImputer(strategy="median")
sc  = StandardScaler()

X_train = sc.fit_transform(imp.fit_transform(train[features]))
X_test  = sc.transform(imp.transform(test[features]))

# ----- FIT HIERARCHICAL CLUSTERING
hc = AgglomerativeClustering(n_clusters=3, linkage="ward")
train["Cluster"] = hc.fit_predict(X_train)

# ----- COMPUTE CLUSTER CENTROIDS
centroids = (
    pd.DataFrame(X_train)
    .assign(Cluster=train["Cluster"].values)
    .groupby("Cluster")
    .mean()
    .values
)

# ----- ASSIGN TEST CLUSTERS
test["Cluster"] = cdist(X_test, centroids).argmin(axis=1)

# ----- LEARN CLUSTER RISKS
cluster_sev_risk = train.groupby("Cluster")[TARGET_SEVERITY].mean()
cluster_mace_risk = train.groupby("Cluster")[TARGET_MACE].mean()

# ----- PREDICT
severity_pred = test["Cluster"].map(cluster_sev_risk)
mace_pred = test["Cluster"].map(cluster_mace_risk)

# ----- SUBMISSION
submission = pd.DataFrame({
    "trustii_id": test[ID_COL],
    "OUTCOME MACE": (mace_pred > 0.5).astype(int),
    "OUTCOME SEVERITY": severity_pred
})

submission = submission[dummy.columns]
submission.to_csv("submission_hierarchical.csv", index=False)

print("✅ submission_hierarchical.csv saved")


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

features = [AGE_COL, SEX_COL] + SNP_COLS

# SEVERITY (proba)
sev_model = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
    ("tree", DecisionTreeClassifier(max_depth=5, random_state=42))
])

sev_model.fit(train[features], train[TARGET_SEVERITY])
severity_pred = sev_model.predict_proba(test[features])[:, 1]

# MACE (classe)
mace_model = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
    ("tree", DecisionTreeClassifier(max_depth=6, random_state=42))
])

mace_model.fit(train[features], train[TARGET_MACE])
mace_pred = mace_model.predict(test[features])

submission = pd.DataFrame({
    "trustii_id": test[ID_COL],
    "OUTCOME MACE": mace_pred.astype(int),
    "OUTCOME SEVERITY": severity_pred.astype(float)
})

submission = submission[dummy.columns]
submission.to_csv("submission_tree.csv", index=False)
print("✅ submission_tree.csv saved. Final shape:", submission.shape)


In [ ]:
from sklearn.cluster import SpectralClustering
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

features = [AGE_COL, SEX_COL] + SNP_COLS

imp = SimpleImputer(strategy="median")
sc  = StandardScaler()

X_test = sc.fit_transform(imp.fit_transform(test[features]))

# SEVERITY (pseudo-score)
sev_labels = SpectralClustering(
    n_clusters=2,
    affinity="nearest_neighbors",
    random_state=42
).fit_predict(X_test)

severity_pred = sev_labels.astype(float)

# MACE (cluster)
mace_pred = SpectralClustering(
    n_clusters=3,
    affinity="nearest_neighbors",
    random_state=42
).fit_predict(X_test)

submission = pd.DataFrame({
    "trustii_id": test[ID_COL],
    "OUTCOME MACE": mace_pred.astype(int),
    "OUTCOME SEVERITY": severity_pred.astype(float)
})

submission = submission[dummy.columns]
submission.to_csv("submission_spectral.csv", index=False)
print("✅ submission_spectral.csv saved. Final shape:", submission.shape)
